In [1]:
import os
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from PIL import Image


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


if torch.backends.mps.is_available():
    device = torch.device("mps")

elif torch.cuda.is_available():
    device = torch.device("cuda")

else:
    device = torch.device("cpu")


print("Device:", device)

Device: mps


In [3]:
from pathlib import Path
import kagglehub

# Project-local dataset folder
DATA_DIR = Path("./dataset")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle dataset handle - replace with the real owner/dataset-slug
DATASET_REF = "OWNER/DATASET-SLUG"  # e.g. 'zynicide/wine-reviews'

# Check for Kaggle credentials (either ~/.kaggle/kaggle.json or env vars)
kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
if not kaggle_json.exists() and not (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')):
    print("Warning: Kaggle credentials not found.")
    print("Place your kaggle.json at ~/.kaggle/kaggle.json or set KAGGLE_USERNAME and KAGGLE_KEY.")
    print("You can also run: `kaggle auth login` or `kaggle config` from the CLI.")

# Try to download the dataset with friendly error handling
from kagglehub.exceptions import KaggleApiHTTPError
import traceback

try:
    DATA_PATH = kagglehub.dataset_download(
        DATASET_REF,
        output_dir=str(DATA_DIR)
    )
except KaggleApiHTTPError as e:
    print("Kaggle API error while downloading dataset:", e)
    resp = getattr(e, 'response', None)
    status = getattr(resp, 'status_code', None)
    if status == 403:
        print("HTTP 403 Forbidden: You likely don't have permission to access this dataset.")
        print("Actions: 1) Verify the handle (owner/dataset-slug). 2) If the dataset is private, ensure you've accepted any required terms on Kaggle. 3) Authenticate with kaggle CLI or provide ~/.kaggle/kaggle.json.")
    else:
        print("HTTP error from Kaggle API (status=", status, "), see traceback below:")
    traceback.print_exc()
    print("Manual alternative: use the kaggle CLI to download: `kaggle datasets download -d owner/dataset-slug -p ./dataset`")
except Exception as e:
    print("Unexpected error while downloading dataset:", e)
    traceback.print_exc()
    print("Manual alternative: use the kaggle CLI to download: `kaggle datasets download -d owner/dataset-slug -p ./dataset`")
else:
    print("Dataset downloaded to:")
    print(DATA_PATH)
    print("\nAbsolute path:")
    print(DATA_DIR.resolve())

Place your kaggle.json at ~/.kaggle/kaggle.json or set KAGGLE_USERNAME and KAGGLE_KEY.
You can also run: `kaggle auth login` or `kaggle config` from the CLI.
Kaggle API error while downloading dataset: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.
HTTP 403 Forbidden: You likely don't have permission to access this dataset.
Actions: 1) Verify the handle (owner/dataset-slug). 2) If the dataset is private, ensure you've accepted any required terms on Kaggle. 3) Authenticate with kaggle CLI or provide ~/.kaggle/kaggle.json.
Manual alternative: use the kaggle CLI to download: `kaggle datasets download -d owner/dataset-slug -p ./dataset`


Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/kagglehub/exceptions.py", line 67, in handle_call
    return fn()
           ^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/kagglehub/http_resolver.py", line 364, in <lambda>
    dataset = handle_call(lambda: api_client.datasets.dataset_api_client.get_dataset(r))
                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/kagglesdk/datasets/services/dataset_api_service.py", line 33, in get_dataset
    return self._client.call("datasets.DatasetApiService", "GetDataset", request, ApiDataset)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/kagglesdk/kaggle_http_client.py", line